# 06 · Schedulers

> **Source notes:** `Schedulers.md`

Swap the sampler → cut generation time by 50× **without retraining**.

This notebook demonstrates:
- Visualise the DDPM noise schedule (ᾱₜ curve)
- Implement DDIM sampling on our Ch.4/Ch.5 `CondUNet`
- Compare image quality vs. step count for DDPM and DDIM
- Profile wall-clock time per step count
- Visualise how timestep sub-sequences differ (uniform vs. SNR-optimal)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `run()` to produce the result
# 2. Process data
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy", "-q"], check=True)

import torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt, time
from torch.utils.data import DataLoader
print("Ready.")

## 1 · Noise Schedule Visualisation

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `T_STEPS`
# 2. Compute `betas_linear` using `schedule()`
# 3. Compute `steps` using `schedule()`
# 4. Plot results -- call `subplots()`
# 5. Compute `t_range` using `plot()`
# 6. Compute `snr_linear` using `semilogy()`
# 7. Plot results -- call `suptitle()`
# 8. Call `region()` to produce the result
#
# Hint:
#    betas_linear = torch.linspace(???)
#    alpha_bar_linear = torch.cumprod(???)
#    steps = torch.arange(???)
#    f = torch.cos(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
T_STEPS = 1000

# Linear schedule (default DDPM)
betas_linear    = torch.linspace(1e-4, 0.02, T_STEPS)
alphas_linear   = 1 - betas_linear
alpha_bar_linear = torch.cumprod(alphas_linear, 0)

# Cosine schedule (improved DDPM, Nichol & Dhariwal 2021)
steps = torch.arange(T_STEPS + 1)
f     = torch.cos((steps / T_STEPS + 0.008) / 1.008 * (torch.pi / 2)) ** 2
alpha_bar_cosine = f / f[0]
alpha_bar_cosine = alpha_bar_cosine[1:]   # same length as linear

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

t_range = torch.arange(T_STEPS)
ax1.plot(t_range, alpha_bar_linear, label="Linear β schedule", color="steelblue")
ax1.plot(t_range, alpha_bar_cosine, label="Cosine schedule", color="tomato", linestyle="--")
ax1.set_xlabel("Timestep t"); ax1.set_ylabel("ᾱₜ (signal fraction²)")
ax1.set_title("Noise Schedule: Linear vs. Cosine")
ax1.legend(); ax1.grid(alpha=0.3)

# SNR = ᾱ / (1-ᾱ) — log scale shows where gradient signal lives
snr_linear = alpha_bar_linear / (1 - alpha_bar_linear + 1e-8)
snr_cosine = alpha_bar_cosine / (1 - alpha_bar_cosine + 1e-8)
ax2.semilogy(t_range, snr_linear, label="Linear", color="steelblue")
ax2.semilogy(t_range, snr_cosine, label="Cosine", color="tomato", linestyle="--")
ax2.set_xlabel("Timestep t"); ax2.set_ylabel("SNR (log scale)")
ax2.set_title("Signal-to-Noise Ratio")
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Cosine schedule gives more uniform SNR across timesteps")
plt.tight_layout(); plt.show()

print("Linear: high-SNR region (t<200) spans only 20% of timesteps")
print("Cosine: SNR transitions more gradually → better use of training budget")

## 2 · Rebuild the Conditional U-Net (from Ch.5)

We copy the model architecture and train quickly (15 epochs) for comparison demos.

In [ ]:
def q_sample(x0, t):
    """
    TODO #3: Implement `q_sample()`.

    Steps:
    1. Compute `betas` using `sqrt()`
    2. Process data
    3. Define helper function
    4. Compute `model` using `CondUNet()`
    5. Define helper function `q_sample()`
    6. Call `train()` to produce the result

    Hint:
    enc1 = ResBlock(???)
    enc2 = ResBlock(???)
    sqrt_ab = alpha_bar.sqrt(???)
    freqs = torch.exp(???)

    Returns: torch.cat([args.sin(), args.cos()], d...
    """
    raise NotImplementedError("TODO: implement q_sample()")

## 3 · DDIM Sampler Implementation

In [ ]:
def ddim_sample(model, classes, n_steps=50, guidance_scale=3.0, eta=0.0):
    """
    TODO #4: Implement `ddim_sample()`.

    Steps:
    1. Define helper function `ddim_sample()`
    2. Call `linspace()` to produce the result
    3. Call `full()` to produce the result
    4. Call `model()` to produce the result
    5. Call `tensor()` to produce the result
    6. Call `clamp()` to produce the result
    7. Call `sqrt()` to produce the result
    8. Define helper function `ddpm_sample()`
    9. Process data

    Hint:
    null = torch.full(???)
    x = torch.randn(???)
    ts = torch.linspace(???)
    tb = torch.full(???)

    Returns: x
    """
    raise NotImplementedError("TODO: implement ddim_sample()")

def ddpm_sample(model, classes, guidance_scale=3.0):
    """
    TODO #4: Implement `ddpm_sample()`.

    Steps:
    1. Define helper function `ddim_sample()`
    2. Call `linspace()` to produce the result
    3. Call `full()` to produce the result
    4. Call `model()` to produce the result
    5. Call `tensor()` to produce the result
    6. Call `clamp()` to produce the result
    7. Call `sqrt()` to produce the result
    8. Define helper function `ddpm_sample()`
    9. Process data

    Hint:
    null = torch.full(???)
    x = torch.randn(???)
    ts = torch.linspace(???)
    tb = torch.full(???)

    Returns: x
    """
    raise NotImplementedError("TODO: implement ddpm_sample()")

## 4 · Speed vs. Quality: Step Count Comparison

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `classes` using `tensor()`
# 2. Compute `results` using `time()`
# 3. Compute `t0` using `time()`
# 4. Plot results -- call `subplots()`
# 5. Plot results -- call `imshow()`
# 6. Plot results -- call `suptitle()`
#
# Hint:
#    classes = torch.tensor(???)
#    t0 = time.time(???)
#    elapsed = time.time(???)
#    ddpm_time = time.time(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
classes = torch.tensor([3, 3, 3, 3])  # generate '3' four times
step_counts = [5, 10, 20, 50, 100]

results = {}
for n in step_counts:
    t0 = time.time()
    imgs = ddim_sample(model, classes, n_steps=n, guidance_scale=3.0)
    elapsed = time.time() - t0
    results[n] = (imgs, elapsed)
    print(f"DDIM {n:4d} steps: {elapsed:.2f}s")

# Also run full DDPM for reference
t0 = time.time()
ddpm_imgs = ddpm_sample(model, classes, guidance_scale=3.0)
ddpm_time = time.time() - t0
print(f"DDPM 1000 steps: {ddpm_time:.2f}s")

# Display
n_cols = len(classes)
all_configs = step_counts + ["DDPM-1000"]
fig, axes = plt.subplots(len(all_configs), n_cols, figsize=(n_cols*2, len(all_configs)*2.2))

for row, config in enumerate(all_configs):
    if config == "DDPM-1000":
        imgs, elapsed = ddpm_imgs, ddpm_time
        label = f"DDPM-1000\n{elapsed:.1f}s"
    else:
        imgs, elapsed = results[config]
        label = f"DDIM-{config}\n{elapsed:.1f}s"
    for col in range(n_cols):
        axes[row, col].imshow(imgs[col, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(label, rotation=0, labelpad=55, va="center", fontsize=8)

plt.suptitle("DDIM step count vs. DDPM 1000-step baseline (digit '3')", y=1.01)
plt.tight_layout(); plt.show()

## 5 · Determinism: Same Seed → Same Image

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `classes_single` using `tensor()`
# 2. Compute `results_det` using `manual_seed()`
# 3. Compute `max_diff` using `item()`
# 4. Compute `results_stoch` using `stochastic()`
# 5. Compute `max_diff_stoch` using `item()`
# 6. Plot results -- call `imshow()`
# 7. Plot results -- call `suptitle()`
#
# Hint:
#    classes_single = torch.tensor(???)
#    axes = plt.subplots(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# DDIM is deterministic: same seed reproduces exactly
classes_single = torch.tensor([7])

results_det = []
for run in range(4):
    torch.manual_seed(42)  # same seed each time
    img = ddim_sample(model, classes_single, n_steps=20, guidance_scale=3.0, eta=0.0)
    results_det.append(img)

# Check they are pixel-identical
max_diff = max(abs(results_det[i] - results_det[0]).max().item() for i in range(1, 4))
print(f"Max pixel difference across 4 runs with same seed: {max_diff:.6f}")

# Now stochastic (eta=1)
results_stoch = []
for run in range(4):
    torch.manual_seed(42)
    img = ddim_sample(model, classes_single, n_steps=20, guidance_scale=3.0, eta=1.0)
    results_stoch.append(img)

# These should differ because of internal randn calls
max_diff_stoch = max(abs(results_stoch[i] - results_stoch[0]).max().item() for i in range(1, 4))

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for col, img in enumerate(results_det):
    axes[0, col].imshow(img[0,0].numpy(), cmap="gray", vmin=-1, vmax=1)
    axes[0, col].set_title(f"Run {col+1}"); axes[0, col].axis("off")
axes[0, 0].set_ylabel("η=0 (det.)", rotation=0, labelpad=50)
for col, img in enumerate(results_stoch):
    axes[1, col].imshow(img[0,0].numpy(), cmap="gray", vmin=-1, vmax=1)
    axes[1, col].set_title(f"Run {col+1}"); axes[1, col].axis("off")
axes[1, 0].set_ylabel("η=1 (stoch.)", rotation=0, labelpad=50)

plt.suptitle(f"Deterministic (η=0): max diff = {max_diff:.2e}  |  Stochastic (η=1): max diff = {max_diff_stoch:.2e}")
plt.tight_layout(); plt.show()

## 6 · Summary

```
Scheduler decision tree:

 Need real-time (<1s)? ──Yes──▶ LCM / SDXL-Turbo (needs distilled model)
 │No
 ▼
 Need highest quality? ──Yes──▶ DPM-Solver++ (15-20 steps)
 │No
 ▼
 Need reproducibility? ──Yes──▶ DDIM (η=0, 20-50 steps)
 │No
 ▼
 Need maximum diversity ──Yes──▶ DDPM or DDIM (η=1, 50+ steps)
```

**Next:** [LatentDiffusion.md](../LatentDiffusion/LatentDiffusion.md) — bring diffusion to real images by first compressing to latent space.